# Demo 04: Power Iteration for Improved Accuracy

This demo shows how power iteration improves sketching accuracy.

Power iteration applies `(A'A)^q` to the random sketch, which amplifies
the dominant singular components. This is especially helpful when:
- The spectral gap is small
- High accuracy is needed
- The matrix has slowly decaying singular values

The demo tests a 2D grid of parameters:
- **extra_samples**: How much oversampling (more = better accuracy)
- **power_iter**: Number of power iterations (more = better accuracy)

Trade-off: More iterations/samples = better accuracy but more computation.

Try changing the **Configuration** parameters below to experiment!

In [1]:
# Configuration - Modify these to experiment

MATRIX_SIZE = (2000, 1000)    # (rows, columns)
TARGET_RANK = 50            # Number of singular values to compute
RANDOM_SEED = 42            # For reproducibility

# Power iteration settings - test grid of values
EXTRA_SAMPLES_LIST = [24, 18, 12, 6, 3]  # Oversampling values to test
POWER_ITER_LIST = [0, 1, 2, 3, 4]        # Power iteration counts to test

# Matrix type: 'structured' (clear spectral gap) or 'random' (no gap)
MATRIX_TYPE = 'structured'

In [ ]:
import numpy as np
import time
import sys
sys.path.insert(0, '..')

from scipy import linalg
from librla import svd_sketch

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

m, n = MATRIX_SIZE
k = TARGET_RANK

print(f"Matrix: {m} x {n}")
print(f"Target rank: {k}")

# Create test matrix
if MATRIX_TYPE == 'structured':
    print("Matrix type: STRUCTURED")
    print("Singular values: logspace(0,-2,k) + logspace(-2,-10,n-k)")
    # Create matrix with decaying spectrum and clear gap at rank k
    U_full = linalg.orth(np.random.randn(m, m))
    V_full = linalg.orth(np.random.randn(n, n))
    s_true = np.concatenate([
        np.logspace(0, -2, k),      # Fast decay in first k singular values
        np.logspace(-2, -10, n-k)   # Slow decay after
    ])
    U = U_full[:, :n]
    A = U @ np.diag(s_true) @ V_full.T
else:
    print("Matrix type: RANDOM (no spectral gap)")
    A = np.random.randn(m, n)
    s_true = linalg.svd(A, compute_uv=False)

# Matrix properties
cond = s_true[0] / s_true[-1]
gap = s_true[k-1] / s_true[k] if k < len(s_true) else float('inf')

print(f"\nSpectral properties:")
print(f"   s[0]     = {s_true[0]:.6e} (largest)")
print(f"   s[{k-1}]   = {s_true[k-1]:.6e} (at target rank)")
print(f"   s[{k}]   = {s_true[k]:.6e} (first neglected)")
print(f"   s[{n-1}] = {s_true[n-1]:.6e} (smallest)")
print(f"   Condition number: {cond:.2e}")
print(f"   Spectral gap at k={k}: {gap:.1f}x")

## Parameter Grid Test

- `power_iter=0` means no power iteration (baseline)
- Each power iteration costs 2 extra matrix-vector products
- `extra_samples` controls oversampling (`block_size = k + extra_samples`)

In [3]:
# Store results in 2D arrays
errors = np.zeros((len(EXTRA_SAMPLES_LIST), len(POWER_ITER_LIST)))
sval_errors = np.zeros((len(EXTRA_SAMPLES_LIST), len(POWER_ITER_LIST)))
s_ref = s_true[:k]

for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    block_size = k + extra_samples
    print(f"\n--- extra_samples = {extra_samples} (block_size = {block_size}) ---")

    for jdx, power_iter in enumerate(POWER_ITER_LIST):
        t0 = time.perf_counter()
        U, s, Vh = svd_sketch(A, rtol=float(k),
                              power_iter=power_iter,
                              extra_samples=extra_samples)
        elapsed = time.perf_counter() - t0

        # Reconstruction error
        A_approx = U @ np.diag(s) @ Vh
        recon_err = np.linalg.norm(A - A_approx, 'fro') / np.linalg.norm(A, 'fro')
        errors[idx, jdx] = recon_err

        # Singular value accuracy
        sval_err = np.linalg.norm(s - s_ref) / np.linalg.norm(s_ref)
        sval_errors[idx, jdx] = sval_err

        print(f"   power_iter={power_iter}: err={recon_err:.2e}, sval_err={sval_err:.2e}, time={elapsed:.4f}s")


--- extra_samples = 24 (block_size = 74) ---
   power_iter=0: err=3.02e-02, sval_err=1.91e-03, time=0.0123s
   power_iter=1: err=2.12e-02, sval_err=6.02e-05, time=0.0135s
   power_iter=2: err=2.12e-02, sval_err=6.55e-06, time=0.0133s
   power_iter=3: err=2.12e-02, sval_err=8.16e-07, time=0.0137s
   power_iter=4: err=2.12e-02, sval_err=2.03e-07, time=0.0158s

--- extra_samples = 18 (block_size = 68) ---
   power_iter=0: err=3.22e-02, sval_err=2.87e-03, time=0.0055s
   power_iter=1: err=2.13e-02, sval_err=1.51e-04, time=0.0080s
   power_iter=2: err=2.12e-02, sval_err=1.23e-05, time=0.0102s
   power_iter=3: err=2.12e-02, sval_err=2.69e-06, time=0.0120s
   power_iter=4: err=2.12e-02, sval_err=5.33e-07, time=0.0137s

--- extra_samples = 12 (block_size = 62) ---
   power_iter=0: err=3.67e-02, sval_err=3.76e-03, time=0.0049s
   power_iter=1: err=2.13e-02, sval_err=1.92e-04, time=0.0065s
   power_iter=2: err=2.12e-02, sval_err=6.99e-05, time=0.0082s
   power_iter=3: err=2.12e-02, sval_err=1.9

## Summary Tables

In [4]:
print("Reconstruction Error")
print("=" * 40)

# Header row
header = "extra_samples |"
for p in POWER_ITER_LIST:
    header += f"  iter={p}  |"
print(header)
print("-" * 14 + "+" + ("-" * 10 + "+") * len(POWER_ITER_LIST))

# Data rows
for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    row = f"{extra_samples:>13} |"
    for jdx in range(len(POWER_ITER_LIST)):
        row += f" {errors[idx, jdx]:.2e} |"
    print(row)

print("\n")
print("Singular Value Error")
print("=" * 40)

# Header row
header = "extra_samples |"
for p in POWER_ITER_LIST:
    header += f"  iter={p}  |"
print(header)
print("-" * 14 + "+" + ("-" * 10 + "+") * len(POWER_ITER_LIST))

# Data rows
for idx, extra_samples in enumerate(EXTRA_SAMPLES_LIST):
    row = f"{extra_samples:>13} |"
    for jdx in range(len(POWER_ITER_LIST)):
        row += f" {sval_errors[idx, jdx]:.2e} |"
    print(row)

print("\nNotes:")
print("  - Power iteration amplifies dominant singular components")
print("  - Extra samples (oversampling) improves subspace capture")

Reconstruction Error
extra_samples |  iter=0  |  iter=1  |  iter=2  |  iter=3  |  iter=4  |
--------------+----------+----------+----------+----------+----------+
           24 | 3.02e-02 | 2.12e-02 | 2.12e-02 | 2.12e-02 | 2.12e-02 |
           18 | 3.22e-02 | 2.13e-02 | 2.12e-02 | 2.12e-02 | 2.12e-02 |
           12 | 3.67e-02 | 2.13e-02 | 2.12e-02 | 2.12e-02 | 2.12e-02 |
            6 | 4.02e-02 | 2.17e-02 | 2.13e-02 | 2.12e-02 | 2.12e-02 |
            3 | 4.73e-02 | 2.17e-02 | 2.13e-02 | 2.13e-02 | 2.12e-02 |


Singular Value Error
extra_samples |  iter=0  |  iter=1  |  iter=2  |  iter=3  |  iter=4  |
--------------+----------+----------+----------+----------+----------+
           24 | 1.91e-03 | 6.02e-05 | 6.55e-06 | 8.16e-07 | 2.03e-07 |
           18 | 2.87e-03 | 1.51e-04 | 1.23e-05 | 2.69e-06 | 5.33e-07 |
           12 | 3.76e-03 | 1.92e-04 | 6.99e-05 | 1.97e-05 | 7.55e-06 |
            6 | 4.65e-03 | 7.21e-04 | 1.09e-04 | 6.39e-05 | 6.16e-05 |
            3 | 5.88e-03 | 9.27e-